In [61]:

# ../data_recorded/exp1/trial_1/data/BA_24_RA_sr4_20M_2460MHz_off_8MHz_t1739963290557.npy
# ../data_recorded/exp1/trial_1/data/BA_24_RA_sr4_20M_2460MHz_off_8MHz_t1739963290557.npy
# ../data_recorded/exp1/trial_1/data/BA_24_RA_sr4_20M_2460MHz_off_8MHz_t1739963290557.npy
# ../data_recorded/exp1/trial_1/data/BA_24_RA_sr4_20M_2460MHz_off_8MHz_t1739963290557.npy
# ../data_recorded/exp1/trial_1/data/BA_24_RA_sr4_20M_2460MHz_off_8MHz_t1739963290557.npy
# ../data_recorded/exp1/trial_1/data/BA_24_RA_sr4_20M_2460MHz_off_8MHz_t1739963290557.npy













total_data = np.load('../data_recorded/exp1/trial_1/data/BA_24_RA_sr4_20M_2460MHz_off_8MHz_t1739963290557.npy', 'r', allow_pickle=True, )[0]
bkg_data = np.load('../data_recorded/exp1/trial_1/ambient/BA_24_RA_sr4_20M_2460MHz_off_8MHz_t1739963230249.npy', 'r', allow_pickle=True, )[0]



ns_samples = 4
ms_samples = ns_samples * 1000

packet_len = 46 * 16 * 4

space_ms = 10

# space_samples = 11 * ms_samples

num_ms = 20

data_section = total_data[0:(num_ms * ms_samples)]
data_section_bkg = bkg_data[0:(num_ms * ms_samples)]



In [60]:
from math import floor
from numpy import ndarray
from numpy.typing import NDArray
from prometheus_client.decorator import append
from pydantic import BaseModel
from pydantic_numpy import np_array_pydantic_annotated_typing
from random import random
from typing import Tuple
from scipy.signal import savgol_filter
from scipy import signal
import math

from matplotlib import pyplot as plt
import numpy as np
from scipy.signal import argrelextrema
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import Span

output_notebook()


Loading BokehJS ...

In [62]:

def min_max_scale(data):
    min_val = np.min(data)
    max_val = np.max(data)
    return (data - min_val) / (max_val - min_val)

In [63]:
TOOLS = "hover,crosshair,pan,wheel_zoom,zoom_in,zoom_out,box_zoom,undo,redo,reset,tap,save,box_select,poly_select,lasso_select,examine,help"

p = figure(width=1600, height=600, tools=TOOLS)


times = np.linspace(0, len(data_section), num=len(data_section))
data = min_max_scale(np.abs(data_section))

# add a line renderer

p.line(times, data, line_width=2, color='#fff', alpha=0.5)
p.line(times, min_max_scale(np.abs(data_section_bkg)), line_width=2, color='#ffffff', )

min_max_avg_bkg = np.average(min_max_scale(np.abs(data_section_bkg)))

p.line(times, np.full(len(times),min_max_avg_bkg) , line_width=2, color='#f70459')



d_sav_10 = savgol_filter(data, 10, 5)
d_sav_100 = savgol_filter(data, 100, 1)
d_sav_1000 = savgol_filter(data, 1000, 1)
# 
# p.line(times, min_max_scale(d_sav_10), line_width=2, color='#0f0')
p.line(times, min_max_scale(d_sav_100), line_width=2, color='#000')
p.line(times, min_max_scale(d_sav_1000), line_width=2, color='#f00')

idx = np.where( min_max_scale(d_sav_1000) > min_max_avg_bkg, 1, 0)

# find the first value that has this
starts = np.where(np.concatenate((np.array(
    [x != idx[i + 1] and x == 0 for i, x in enumerate(idx[:-1])]), [False])))[0]
ends = np.where(np.concatenate((np.array(
    [x != idx[i + 1] and x == 1 for i, x in enumerate(idx[:-1])]), [False])))[0]

max_packet_fudge =10000# packet_len * 1.10

actual_packets = np.array(
    [(start, end) for start, end in zip(starts, ends) if max_packet_fudge > (end - start) >= packet_len])

# print(actual_packets)

# p.line(times, idx, line_width=2, color='#0f0')
# 
for (start, end) in actual_packets:
    print(end-start-packet_len)
    high_avg = np.average(data[start:end])
    high_avg_avg = np.average(min_max_scale(d_sav_100)[start:end])
    p.line(times[start:end],  np.full(len(times[start:end]),high_avg), line_width=2, color='#0f0')
    p.line(times[start:end],  np.full(len(times[start:end]),high_avg_avg), line_width=2, color='#0ff')

    left_edge = Span(location=times[start], dimension='height',
                     line_color='#0f0', line_width=2)
    p.add_layout(left_edge)
    
    left_edge = Span(location=times[end], dimension='height',
                     line_color='#0f0', line_width=2)
    p.add_layout(left_edge)
    middle = (high_avg_avg + min_max_avg_bkg)/2
    print(high_avg_avg,middle,  min_max_avg_bkg)
    p.line(times[start:end],  np.full(len(times[start:end]),middle), line_width=2, color='#ff0')

    above_middle = np.array([x for x in range(start, end) if d_sav_10[x] > middle])
    start, end = above_middle[0], above_middle[-1]
    print(start, end)




    



show(p)

773
0.6419293 0.4344002 0.22687109
19353 22405
710
0.6514527 0.4391619 0.22687109
64987 68246
